# ROSP-7 diagnostic: the optics contribution to sDIQ and what the AOS was converging to

Companion to `rosp-07_sdiq_from_atm_seeing.ipynb` and `rosp-07_sdiq_from_components.ipynb` for PTH Tier 1 metric
[ROSP-7](https://rubinobs.atlassian.net/browse/ROSP-7). The components notebook uses `aos_fwhm` from
`visit1_quicklook` as the optical-figure term and checks it against the quadrature difference of the delivered
PSF and the donut blur, $\theta_\mathrm{alt}^2 = \mathrm{FWHM}^2 - \mathrm{donut\_blur}^2$. Over the campaign
`aos_fwhm` goes from being *smaller* than $\theta_\mathrm{alt}$ to being *larger*. This notebook asks two things:

1. **Is that crossover driven by the delivered FWHM or by the donut blur?** The two inputs are followed night by
   night and period by period, the change in $\mathrm{FWHM}^2 - \mathrm{donut\_blur}^2$ between periods is split
   into the part from $\mathrm{FWHM}^2$ and the part from $\mathrm{donut\_blur}^2$, and the relation is also
   examined at fixed donut blur (i.e. at fixed atmosphere) so a change in the seeing distribution is not mistaken
   for a change in the optics.
2. **Is something deeper going on with `aos_fwhm`, tied to where the closed loop was told to converge?** The
   AOS converged the corner defocus $z_4$ to a target that changed twice: 0 µm on every corner from
   2025-10-25, −0.2 µm on every corner from 2025-12-13, and per-corner values from 2026-04-15 (R00 −0.1389,
   R40 −0.0328, R04 −0.0986, R44 −0.1447 µm; `z4_targets`). `aos_fwhm` is **recomputed from the per-corner
   Zernikes in the ConsDB** (`ccdvisit1_quicklook.z4 … z28` on the eight wavefront sensors) in several ways:
   relative to the target in force at the time (how well the optics *thought* it was doing), relative to the
   final post-April target everywhere (`reference_target_dayobs`), relative to a zero target, and with the
   defocus term left out altogether. If measuring against the final target makes the recomputed optics term
   line up with $\theta_\mathrm{alt}$ in every period, the crossover is explained by the moving target.

**How `aos_fwhm` is built** (copied from `lsst.rubintv.production.aosRecipes.computeAosResidualFwhm`, the code
that fills the ConsDB column): for each wavefront sensor the measured Zernikes $z_k$ (µm, $k = 4 \ldots 28$) have
the design (intrinsic) Zernikes of that field position, band and rotator angle subtracted, then the per-sensor
offset vector $y_2$ (which is where the $z_4$ target lives: `ts_ofc/policy/y2/lsst_y2.yaml`), and the residual
is converted to a quadrature PSF width with the `ts_wep` factors $c_k$ (arcsec/µm, `convertZernikesToPsfWidth`):

$$ \mathrm{RSS}_\mathrm{sensor} = \sqrt{\sum_k \left[c_k\,(z_k - z_k^\mathrm{intr} - y_{2,k})\right]^2}, \qquad
   \mathrm{aos\_fwhm} = 1.06\,\ln\!\left(1 + \overline{\mathrm{RSS}}_\mathrm{sensors}\right) $$

The logarithm is the empirical adjustment used by Rapid Analysis (`aos_log_adjustment`). Only $y_{2,4}$ is
non-zero, so the variants above differ only in what is subtracted from $z_4$; the higher-order modes are the
same in all of them. The intrinsic Zernikes are evaluated exactly as `lsst.ts.ofc` does (double-Zernike tables
from a pinned `ts_ofc` commit, evaluated with `galsim` at the sensor field position rotated by the rotator
angle), and the stored `aos_fwhm` is used to check the convention: the sign of the rotation and whether the
intrinsics are subtracted at all are tried and the choice that best reproduces the stored column is kept
(`intrinsic_rotation_sign = "auto"`). The recomputed value relative to the period target should then match the
stored column visit by visit; where it does not, the stored column was computed with a different target or a
different pipeline version, and the notebook says so.

Nothing here needs an atmospheric seeing monitor; the atmosphere enters only through the donut blur. The donut
blur is unusable in the y band, so y is excluded (`exclude_bands`). Every number quoted for a set of visits is the
square root of the median of the squared quantity (signed square root when negative), as in the other two
notebooks. Visit-by-visit scatter plots are included for every AOS quantity across the periods.

In [ ]:
# Parameters cell. Set defaults here - gets replaced by the sidecar on Times Square.
start_dayobs = 20251025  # First dayObs (inclusive)
end_dayobs = 20260715  # Last dayObs (inclusive)
science_programs = "407, 408, 416, 417, 419, 421"  # bare numbers become BLOCK-<n>
z4_targets = "20251025: R00=0 R04=0 R40=0 R44=0; 20251213: R00=-0.2 R04=-0.2 R40=-0.2 R44=-0.2; 20260415: R00=-0.1389 R40=-0.0328 R04=-0.0986 R44=-0.1447"  # z4 target (um) per corner, by first dayObs
reference_target_dayobs = (
    20260415  # the target set used to recompute aos_fwhm over the whole range
)
subtract_intrinsics = (
    True  # subtract the design (intrinsic) Zernikes before converting to a PSF width
)
intrinsic_rotation_sign = "auto"  # "+1", "-1" or "auto": sign of the rotator angle used to evaluate the intrinsics
aos_log_adjustment = (
    True  # apply the 1.06 ln(1 + x) adjustment that Rapid Analysis applies to aos_fwhm
)
max_noll_index = 28  # highest Zernike used (ConsDB holds z4..z28)
exclude_bands = "y"  # bands dropped everywhere (the donut blur is unusable in y)
use_cache = (
    True  # cache the ConsDB pulls as parquet under ./cache (ignored on Times Square)
)

not_times_square = True

In [ ]:
# Times Square runs this notebook in a Nublado pod where rubin_nights may be
# missing or stale; install/upgrade it there. Anywhere else this cell is a no-op.
try:
    not_times_square
except NameError:
    not_times_square = False

if not not_times_square:
    print("updating rubin_nights")
    !pip install --user --upgrade git+https://github.com/lsst-sims/rubin_nights.git --no-deps > /dev/null 2>&1

In [ ]:
import inspect
import tempfile
import urllib.request
import os
import re
from dataclasses import dataclass
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, LogNorm
from astropy.time import Time

%matplotlib inline
pd.set_option("display.width", 160)
NOTEBOOK_DIR = os.getcwd()

## Constants, parameter helpers, ConsDB access

The first cells are the shared ones of the companion notebooks (the hub rule is no imports between notebooks),
followed by the constants this notebook adds: the wavefront-sensor detectors, their field positions, the
`ts_ofc` configuration used to evaluate the intrinsic Zernikes, and the `ts_wep` Zernike-to-PSF-width factors.

In [ ]:
DEFAULT_PROGRAMS = (
    "BLOCK-407",
    "BLOCK-408",
    "BLOCK-416",
    "BLOCK-417",
    "BLOCK-419",
    "BLOCK-421",
)

# Bump when the ConsDB query changes so parquet caches written with the old column set are not reused.
CACHE_VERSION = "v2"

SIGMA_TO_FWHM = 2.0 * np.sqrt(2.0 * np.log(2.0))
PLATESCALE = 0.2  # arcsec / pixel, LSSTCam
ARCSEC_TO_RAD = np.pi / (180.0 * 3600.0)
REFERENCE_WAVELENGTH_NM = 500.0
CAM_FWHM = 0.207  # arcsec, camera diffusion term used in image_quality_trending (reference line only)

# Effective wavelengths used by lsst.summit.utils.getBandpassSeeingCorrection for LSSTCam.
LSSTCAM_EFFECTIVE_WAVELENGTH_NM = {
    "u": 365.1397,
    "g": 475.6104,
    "r": 618.6055,
    "i": 753.5490,
    "z": 868.0935,
    "y": 972.5979,
}
BAND_ORDER = ("u", "g", "r", "i", "z", "y")

# Columns pulled from cdb_lsstcam.visit1 (per-visit metadata) and visit1_quicklook (image quality).
VISIT_COLUMNS = (
    "visit_id",
    "day_obs",
    "seq_num",
    "band",
    "physical_filter",
    "science_program",
    "img_type",
    "target_name",
    "exp_time",
    "exp_midpt",
    "exp_midpt_mjd",
    "airmass",
    "altitude",
    "azimuth",
    "zenith_distance",
    "air_temp",
    "pressure",
    "humidity",
    "wind_speed",
    "wind_dir",
    "dimm_seeing",
)
QUICKLOOK_COLUMNS = (
    "psf_sigma_median",
    "psf_sigma_min",
    "psf_sigma_max",
    "n_psf_star_total",
    "aos_fwhm",
    "donut_blur_fwhm",
    "ringss_seeing",
)

# Columns pulled from cdb_lsstcam.exposure (joined on exposure_id = visit_id, the 1-to-1 visit system).
EXPOSURE_COLUMNS = ("observation_reason",)

# RINGSS event in the EFD.
RINGSS_TOPIC = "lsst.sal.ESS.logevent_ringssMeasurement"
RINGSS_SAL_INDEX = 304
RINGSS_FIELDS = (
    "timestamp",
    "hrNum",
    "zenithDistance",
    "flux",
    "fwhmScintillation",
    "fwhmSector",
    "fwhmFree",
    "wind",
)

# Chart palette: fixed categorical order (blue, orange, aqua, yellow, magenta, green, violet, red).
SERIES_COLORS = (
    "#2a78d6",
    "#eb6834",
    "#1baf7a",
    "#eda100",
    "#e87ba4",
    "#008300",
    "#4a3aa7",
    "#e34948",
)
SERIES_MARKERS = ("o", "s", "^", "D", "v", "P", "X", "*")
INK, INK_SECONDARY, INK_MUTED, GRID, SURFACE = (
    "#0b0b0b",
    "#52514e",
    "#898781",
    "#e1e0d9",
    "#fcfcfb",
)
GOAL_COLOR = "#1baf7a"
SEQUENTIAL_BLUE = LinearSegmentedColormap.from_list(
    "seq_blue",
    [
        "#ffffff",
        "#cde2fb",
        "#9ec5f4",
        "#6da7ec",
        "#3987e5",
        "#256abf",
        "#1c5cab",
        "#104281",
        "#0d366b",
    ],
)

plt.rcParams.update(
    {
        "figure.facecolor": SURFACE,
        "axes.facecolor": SURFACE,
        "axes.edgecolor": "#c3c2b7",
        "axes.labelcolor": INK,
        "axes.grid": True,
        "grid.color": GRID,
        "grid.linewidth": 0.8,
        "axes.axisbelow": True,
        "xtick.color": INK_SECONDARY,
        "ytick.color": INK_SECONDARY,
        "legend.frameon": False,
        "font.size": 11,
        "axes.titlesize": 12,
        "lines.linewidth": 2.0,
        "savefig.dpi": 150,
    }
)


def parse_day_obs(value) -> int:
    """Coerce 20260101, "20260101" or "2026-01-01" to an int."""
    s = str(value).strip().replace("-", "")
    if not re.fullmatch(r"\d{8}", s):
        raise ValueError(f"Not a day_obs: {value!r}")
    return int(s)


def parse_programs(value) -> list:
    """List or comma/space separated string of block names; bare numbers become BLOCK-<n>."""
    if value is None:
        return list(DEFAULT_PROGRAMS)
    items = (
        [v for v in re.split(r"[,\s]+", value.strip()) if v]
        if isinstance(value, str)
        else [str(v).strip() for v in value]
    )
    return [f"BLOCK-{it}" if re.fullmatch(r"T?\d+", it) else it for it in items]


def parse_bool(value) -> bool:
    return str(value).strip().lower() in ("true", "1", "yes")


def day_obs_chunks(day_obs_min: int, day_obs_max: int, chunk_days: int):
    """Yield (lo, hi) day_obs pairs covering the range in calendar chunks."""
    lo, end = pd.Timestamp(str(day_obs_min)), pd.Timestamp(str(day_obs_max))
    while lo <= end:
        hi = min(lo + pd.Timedelta(days=chunk_days - 1), end)
        yield int(lo.strftime("%Y%m%d")), int(hi.strftime("%Y%m%d"))
        lo = hi + pd.Timedelta(days=1)


def day_obs_window(day_obs_lo: int, day_obs_hi: int):
    """UTC start/end (astropy Time) covering the nights day_obs_lo..day_obs_hi: 12:00 UTC to 12:00 UTC next day."""
    t0 = datetime.strptime(str(day_obs_lo), "%Y%m%d").replace(
        hour=12, tzinfo=timezone.utc
    )
    t1 = datetime.strptime(str(day_obs_hi), "%Y%m%d").replace(
        hour=12, tzinfo=timezone.utc
    ) + pd.Timedelta(days=1)
    return Time(t0), Time(t1)


def _read_cache(cache_path, use_cache):
    if cache_path and use_cache and os.path.exists(cache_path):
        df = pd.read_parquet(cache_path)
        print(f"Loaded {len(df)} rows from cache {cache_path}")
        return df
    return None


def _write_cache(cache_path, df):
    if cache_path and len(df):
        os.makedirs(os.path.dirname(os.path.abspath(cache_path)), exist_ok=True)
        df.to_parquet(cache_path, index=False)
        print(f"Saved {len(df)} rows to {cache_path}")
    return df


def cached(cache_path, loader, use_cache=True) -> pd.DataFrame:
    """Run loader() unless cache_path already holds its parquet output."""
    df = _read_cache(cache_path, use_cache)
    return df if df is not None else _write_cache(cache_path, loader())


async def cached_async(cache_path, loader, use_cache=True) -> pd.DataFrame:
    """Same as cached() for a loader that returns a coroutine (the EFD client is async on the RSP)."""
    df = _read_cache(cache_path, use_cache)
    return df if df is not None else _write_cache(cache_path, await loader())


# ---- additions for the optics notebook -------------------------------------------------------------------
CACHE_VERSION = "v3-optics"
QUICKLOOK_COLUMNS = QUICKLOOK_COLUMNS + ("physical_rotator_angle",)

# Wavefront sensors (ConsDB ccdvisit1.detector -> name) and their field positions (deg) from ts_ofc
# policy/sample_points/lsst_points.yaml.
CORNER_DETECTORS = {
    191: "R00_SW0",
    192: "R00_SW1",
    195: "R04_SW0",
    196: "R04_SW1",
    199: "R40_SW0",
    200: "R40_SW1",
    203: "R44_SW0",
    204: "R44_SW1",
}
CORNERS = ("R00", "R04", "R40", "R44")
SENSOR_FIELD_DEG = {
    "R00_SW0": (-1.1902777777777778, -1.1902777777777778),
    "R00_SW1": (-1.2549166666666665, -1.1903333333333332),
    "R04_SW0": (-1.1902777777777778, 1.1902777777777778),
    "R04_SW1": (-1.1903333333333332, 1.2549166666666665),
    "R40_SW0": (1.1902777777777778, -1.1902777777777778),
    "R40_SW1": (1.1903333333333332, -1.2549166666666665),
    "R44_SW0": (1.1902777777777778, 1.1902777777777778),
    "R44_SW1": (1.2549166666666665, 1.1903333333333332),
}
Z_MIN = 4
Z_COLUMNS = tuple(f"z{j}" for j in range(Z_MIN, 29))  # ConsDB holds z4..z28 (um)

# ts_ofc configuration (policy/configurations/lsst.yaml) and the pinned commit whose intrinsic tables are used.
TS_OFC_COMMIT = "68f147a989e0f6d1abf28522b2eeae2fbcb23214"  # develop, 2026-09-02
TS_OFC_INTRINSIC_URL = (
    "https://raw.githubusercontent.com/lsst-ts/ts_ofc/"
    + TS_OFC_COMMIT
    + "/python/lsst/ts/ofc/policy/intrinsic_zernikes/lsst/intrinsic_zk_{band}_31_29.yaml"
)
OFC_FIELD_RADIUS_DEG = (0.0, 1.75)  # inner, outer
OFC_PUPIL_RADIUS_M = (2.558, 4.18)  # inner, outer
OFC_WAVELENGTH_UM = {
    "u": 0.365,
    "g": 0.480,
    "r": 0.622,
    "i": 0.754,
    "z": 0.869,
    "y": 0.971,
}

# ts_wep convertZernikesToPsfWidth factors (arcsec per um of Zernike amplitude) for Noll 4..28, LSST pupil
# (8.36 m, obscuration 0.612), computed with galsim 2.8.5; recomputed and checked below when galsim is available.
PSF_WIDTH_PER_ZERNIKE = np.array(
    [
        0.754536,
        0.271131,
        0.271131,
        0.822372,
        0.822372,
        0.395725,
        0.395725,
        1.687193,
        0.939566,
        0.939566,
        0.516431,
        0.516431,
        1.762557,
        1.762557,
        1.091120,
        1.091120,
        0.634981,
        0.634981,
        2.823214,
        1.875882,
        1.875882,
        1.264374,
        1.264374,
        0.752412,
        0.752412,
    ]
)
AOS_LOG_FACTOR = 1.06  # aos_fwhm = 1.06 ln(1 + mean RSS), as in lsst.rubintv.production

PERIOD_COLORS = (
    SERIES_COLORS[0],
    SERIES_COLORS[1],
    SERIES_COLORS[2],
    SERIES_COLORS[3],
    SERIES_COLORS[6],
)
VARIANT_STYLE = {  # recomputed aos_fwhm variants: label, colour, marker, linestyle
    "aos_fwhm": ("aos_fwhm (ConsDB)", INK, "o", "-"),
    "aos_period": ("recomputed: period target", SERIES_COLORS[0], "s", "-"),
    "aos_reference": ("recomputed: reference target", SERIES_COLORS[1], "^", "-"),
    "aos_zero": ("recomputed: zero target", SERIES_COLORS[3], "D", ":"),
    "aos_noz4": ("recomputed: without z4", SERIES_COLORS[6], "v", "--"),
}

In [ ]:
USDF_CONSDB_URL = "http://consdb-pq.consdb:8080/consdb"


class _SummitUtilsClient:
    """Adapter so lsst.summit.utils.ConsDbClient.query returns a DataFrame."""

    def __init__(self, url=USDF_CONSDB_URL):
        from lsst.summit.utils import ConsDbClient

        os.environ["no_proxy"] = os.environ.get("no_proxy", "") + ",.consdb"
        self._client = ConsDbClient(url)

    def query(self, sql):
        result = self._client.query(sql)
        return result if isinstance(result, pd.DataFrame) else result.to_pandas()


def get_consdb_client():
    """Object with .query(sql) -> DataFrame: rubin_nights first, then summit_utils."""
    errors = []
    try:
        from rubin_nights import connections

        client = connections.get_clients()["consdb"]
        print("ConsDB client: rubin_nights")
        return client
    except Exception as exc:
        errors.append(f"rubin_nights: {exc!r}")
    try:
        client = _SummitUtilsClient()
        print("ConsDB client: lsst.summit.utils")
        return client
    except Exception as exc:
        errors.append(f"summit_utils: {exc!r}")
    raise RuntimeError("Could not build a ConsDB client:\n  " + "\n  ".join(errors))


def _sql_list(values):
    return ", ".join("'" + str(v).replace("'", "''") + "'" for v in values)


def build_query(
    day_obs_min,
    day_obs_max,
    programs=DEFAULT_PROGRAMS,
    instrument="lsstcam",
    img_types=("science",),
):
    """SQL joining visit1 (pointing, weather, DIMM), visit1_quicklook (PSF stats) and exposure (observation_reason), one row per visit."""
    schema = f"cdb_{instrument}"
    where = [f"v.day_obs >= {int(day_obs_min)}", f"v.day_obs <= {int(day_obs_max)}"]
    if programs:
        where.append(f"v.science_program IN ({_sql_list(programs)})")
    if img_types:
        where.append(f"v.img_type IN ({_sql_list(img_types)})")
    return (
        "SELECT\n    "
        + ",\n    ".join(
            [f"v.{c}" for c in VISIT_COLUMNS]
            + [f"q.{c}" for c in QUICKLOOK_COLUMNS]
            + [f"e.{c}" for c in EXPOSURE_COLUMNS]
        )
        + f"\nFROM {schema}.visit1 AS v"
        + f"\nLEFT JOIN {schema}.visit1_quicklook AS q ON q.visit_id = v.visit_id"
        + f"\nLEFT JOIN {schema}.exposure AS e ON e.exposure_id = v.visit_id"
        + "\nWHERE "
        + "\n  AND ".join(where)
        + "\nORDER BY v.day_obs, v.seq_num"
    )


def fetch_visits(
    client, day_obs_min, day_obs_max, programs=DEFAULT_PROGRAMS, chunk_days=31
):
    frames = []
    for lo, hi in day_obs_chunks(day_obs_min, day_obs_max, chunk_days):
        df = client.query(build_query(lo, hi, programs))
        print(f"  consdb {lo}-{hi}: {len(df):6d} visits")
        if len(df):
            frames.append(df)
    if not frames:
        return pd.DataFrame(
            columns=list(VISIT_COLUMNS)
            + list(QUICKLOOK_COLUMNS)
            + list(EXPOSURE_COLUMNS)
        )
    out = pd.concat(frames, ignore_index=True).sort_values(["day_obs", "seq_num"])
    missing = [
        c for c in VISIT_COLUMNS + QUICKLOOK_COLUMNS + EXPOSURE_COLUMNS if c not in out
    ]
    if missing:
        print(f"WARNING: columns not returned by the ConsDB: {missing}")
    for c in out.columns:
        if (
            c
            not in (
                "band",
                "physical_filter",
                "science_program",
                "observation_reason",
                "img_type",
                "target_name",
                "exp_midpt",
            )
            and out[c].dtype == object
        ):
            out[c] = pd.to_numeric(out[c], errors="coerce")
    return out.reset_index(drop=True)


def build_corner_query(
    day_obs_min,
    day_obs_max,
    programs=DEFAULT_PROGRAMS,
    instrument="lsstcam",
    detectors=tuple(CORNER_DETECTORS),
    img_types=("science",),
):
    """SQL: one row per (visit, wavefront sensor) with z4..z28 from ccdvisit1_quicklook."""
    schema = f"cdb_{instrument}"
    where = [
        f"v.day_obs >= {int(day_obs_min)}",
        f"v.day_obs <= {int(day_obs_max)}",
        f"c.detector IN ({', '.join(str(int(d)) for d in detectors)})",
    ]
    if programs:
        where.append(f"v.science_program IN ({_sql_list(programs)})")
    if img_types:
        where.append(f"v.img_type IN ({_sql_list(img_types)})")
    return (
        "SELECT\n    c.visit_id,\n    c.detector,\n    "
        + ",\n    ".join(f"q.{c}" for c in Z_COLUMNS)
        + f"\nFROM {schema}.ccdvisit1 AS c"
        + f"\nJOIN {schema}.ccdvisit1_quicklook AS q ON q.ccdvisit_id = c.ccdvisit_id"
        + f"\nJOIN {schema}.visit1 AS v ON v.visit_id = c.visit_id"
        + "\nWHERE "
        + "\n  AND ".join(where)
        + "\nORDER BY c.visit_id, c.detector"
    )


def fetch_corner_zernikes(
    client, day_obs_min, day_obs_max, programs=DEFAULT_PROGRAMS, chunk_days=31
):
    frames = []
    for lo, hi in day_obs_chunks(day_obs_min, day_obs_max, chunk_days):
        df = client.query(build_corner_query(lo, hi, programs))
        print(f"  consdb corners {lo}-{hi}: {len(df):7d} sensor rows")
        if len(df):
            frames.append(df)
    if not frames:
        return pd.DataFrame(columns=["visit_id", "detector", *Z_COLUMNS])
    out = pd.concat(frames, ignore_index=True)
    for c in out.columns:
        if out[c].dtype == object:
            out[c] = pd.to_numeric(out[c], errors="coerce")
    out["detector"] = out["detector"].astype(int)
    out["sensor"] = out["detector"].map(CORNER_DETECTORS)
    out["corner"] = out["sensor"].str[:3]
    return out.sort_values(["visit_id", "detector"]).reset_index(drop=True)

## Physics: targets, intrinsic Zernikes, Zernike residual to PSF width

`parse_z4_targets` turns the `z4_targets` string into a list of (first dayObs, {corner: z4 target}); each visit is
assigned the set in force on its night (`period`). `intrinsic_zernikes` reproduces `lsst.ts.ofc`'s
`get_intrinsic_zernikes`: the double-Zernike table of the band, evaluated at the sensor field position rotated by
the rotator angle, times the band wavelength. `psf_width_per_zernike` is `ts_wep`'s `convertZernikesToPsfWidth`.
`corner_residuals` then forms, per sensor row, the RSS for every target variant and `aggregate_corners` averages
the sensors of each visit into the recomputed `aos_fwhm` values (`aos_period`, `aos_reference`, `aos_zero`,
`aos_noz4`), together with the per-corner $z_4$ deviation (`z4dev_R00`, ...) and the RSS split into its $z_4$ and
higher-order parts.

In [ ]:
def signed_sqrt(x):
    x = np.asarray(x, dtype=float)
    return np.sign(x) * np.sqrt(np.abs(x))


def parse_bands(value):
    if value is None:
        return ()
    items = [v for v in re.split(r"[,\s]+", str(value).strip()) if v]
    return tuple(
        it.strip()[0] for it in items if it.strip().lower() not in ("none", "")
    )


def parse_z4_targets(spec):
    """'20251025: R00=0 R04=0 R40=0 R44=0; 20251213: ...' -> [(dayobs, {corner: z4_um}), ...] sorted by dayobs."""
    out = []
    for chunk in [c for c in re.split(r";", str(spec)) if c.strip()]:
        head, _, body = chunk.partition(":")
        day_obs = parse_day_obs(head)
        vals = {}
        for item in re.findall(
            r"(R\d\d)\s*=\s*([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)", body
        ):
            vals[item[0]] = float(item[1])
        missing = [c for c in CORNERS if c not in vals]
        if missing:
            raise ValueError(
                f"z4 target for {day_obs} lacks corners {missing}: {chunk!r}"
            )
        out.append((day_obs, {c: vals[c] for c in CORNERS}))
    if not out:
        raise ValueError(f"No z4 targets parsed from {spec!r}")
    return sorted(out, key=lambda t: t[0])


def target_label(day_obs, vals, index=None):
    """Short period name, e.g. 'P1 2025-12-13: z4 -0.2' ('z4 per corner' for a per-corner set; see target_description)."""
    uniq = sorted(set(vals.values()))
    body = f"z4 {uniq[0]:g} um" if len(uniq) == 1 else "z4 per corner"
    prefix = f"P{index} " if index is not None else ""
    return f"{prefix}{pd.Timestamp(str(day_obs)).strftime('%Y-%m-%d')}: {body}"


def target_description(day_obs, vals):
    """Full description of a target set: 'from 20260415: R00 -0.1389 R04 -0.0986 R40 -0.0328 R44 -0.1447 um'."""
    return f"from {day_obs}: " + " ".join(f"{c} {v:g}" for c, v in vals.items()) + " um"


def assign_periods(day_obs, targets):
    """Index into targets of the set in force on each night (nights before the first set use the first set)."""
    starts = np.array([t[0] for t in targets])
    idx = np.searchsorted(starts, np.asarray(day_obs, dtype=int), side="right") - 1
    return np.clip(idx, 0, len(targets) - 1)


def psf_width_per_zernike(jmin=Z_MIN, jmax=28, diameter=8.36, obscuration=0.612):
    """ts_wep convertZernikesToPsfWidth factors (arcsec/um) for Noll jmin..jmax; galsim when available, else the table."""
    try:
        import galsim
    except Exception:
        return PSF_WIDTH_PER_ZERNIKE[: jmax - jmin + 1].copy()
    out = np.zeros(jmax + 1)
    r_outer, r_inner = diameter / 2, diameter / 2 * obscuration
    for j in range(jmin, jmax + 1):
        zk = galsim.zernike.Zernike([0] * j + [1], R_outer=r_outer, R_inner=r_inner)
        rms = np.sqrt(np.sum(zk.gradX.coef**2 + zk.gradY.coef**2) / 2)
        out[j] = SIGMA_TO_FWHM * np.rad2deg(rms * 1e-6) * 3600
    return out[jmin:]


def load_intrinsic_tables(bands, cache_dir=None):
    """{band: (31, 29) double-Zernike coefficients} from the pinned ts_ofc policy files (downloaded once, cached)."""
    import yaml

    folder = cache_dir or os.path.join(tempfile.gettempdir(), "ts_ofc_intrinsics")
    os.makedirs(folder, exist_ok=True)
    tables = {}
    for band in bands:
        path = os.path.join(folder, f"intrinsic_zk_{band}_31_29.yaml")
        if not os.path.exists(path):
            url = TS_OFC_INTRINSIC_URL.format(band=band)
            print(f"  downloading {url}")
            with urllib.request.urlopen(url, timeout=60) as resp, open(
                path, "wb"
            ) as fh:
                fh.write(resp.read())
        tables[band] = np.array(yaml.safe_load(open(path)), dtype=float)
    return tables


def intrinsic_zernikes(tables, band, sensor, rot_deg, jmax=28, rotation_sign=1.0):
    """Intrinsic z4..zjmax (um) for every element of rot_deg, as lsst.ts.ofc.utils.get_intrinsic_zernikes.

    The sensor field position is rotated by -rotation_sign * rot_deg before the double-Zernike table of the band
    is evaluated there (galsim zernikeBasis on the field annulus), and the result is scaled by the band wavelength.
    """
    import galsim

    coef = tables[band]
    rot = np.deg2rad(-rotation_sign * np.asarray(rot_deg, dtype=float))
    fx0, fy0 = SENSOR_FIELD_DEG[sensor]
    fx = fx0 * np.cos(rot) + fy0 * np.sin(rot)
    fy = -fx0 * np.sin(rot) + fy0 * np.cos(rot)
    basis = galsim.zernike.zernikeBasis(
        coef.shape[0] - 1,
        fx,
        fy,
        R_outer=OFC_FIELD_RADIUS_DEG[1],
        R_inner=OFC_FIELD_RADIUS_DEG[0],
    )  # (kmax + 1, n)
    pupil = basis.T @ coef  # (n, jmax_table + 1), Noll index along axis 1
    out = np.full((len(fx), jmax - Z_MIN + 1), np.nan)
    n_avail = min(jmax, coef.shape[1] - 1) - Z_MIN + 1
    out[:, :n_avail] = pupil[:, Z_MIN : Z_MIN + n_avail] * OFC_WAVELENGTH_UM[band]
    return out


def corner_residuals(
    corners,
    visits,
    targets,
    reference_index,
    tables=None,
    rotation_sign=1.0,
    subtract_intrinsics=True,
    jmax=28,
):
    """Per sensor row: z4 deviation and the RSS PSF width (arcsec) for each target variant.

    Adds zdev_<k> (measured minus intrinsic, um), z4_target_period / z4_target_reference (um), rss_period,
    rss_reference, rss_zero (target 0), rss_noz4 (higher-order modes only) and rss_z4_period (the z4 term alone).
    """
    cols = [f"z{j}" for j in range(Z_MIN, jmax + 1)]
    factors = psf_width_per_zernike(Z_MIN, jmax)
    out = corners.merge(
        visits[["visit_id", "day_obs", "band", "physical_rotator_angle"]],
        on="visit_id",
        how="inner",
    )
    z = out[cols].to_numpy(dtype=float)
    intr = np.zeros_like(z)
    if subtract_intrinsics:
        if tables is None:
            raise ValueError("subtract_intrinsics needs the intrinsic tables")
        rot = out["physical_rotator_angle"].to_numpy(dtype=float)
        for (band, sensor), idx in out.groupby(["band", "sensor"]).indices.items():
            b = str(band).strip()[0]
            if b not in tables:
                intr[idx] = np.nan
                continue
            intr[idx] = intrinsic_zernikes(
                tables, b, sensor, rot[idx], jmax=jmax, rotation_sign=rotation_sign
            )
    dev = z - intr
    for k, c in enumerate(cols):
        out[f"zdev_{c[1:]}"] = dev[:, k]
    period = assign_periods(out["day_obs"].to_numpy(), targets)
    out["period"] = period
    out["z4_target_period"] = [targets[p][1][c] for p, c in zip(period, out["corner"])]
    out["z4_target_reference"] = out["corner"].map(targets[reference_index][1])
    hi2 = np.nansum((factors[1:] * dev[:, 1:]) ** 2, axis=1)
    hi2 = np.where(np.isfinite(dev[:, 1:]).any(axis=1), hi2, np.nan)
    out["rss_noz4"] = np.sqrt(hi2)
    for name, target in (
        ("period", out["z4_target_period"].to_numpy()),
        ("reference", out["z4_target_reference"].to_numpy()),
        ("zero", np.zeros(len(out))),
    ):
        z4_term = factors[0] * (dev[:, 0] - target)
        out[f"rss_{name}"] = np.sqrt(z4_term**2 + hi2)
        if name == "period":
            out["rss_z4_period"] = np.abs(z4_term)
    out["rss_z4_reference"] = np.abs(
        factors[0] * (dev[:, 0] - out["z4_target_reference"].to_numpy())
    )
    return out


def aos_from_rss(mean_rss, log_adjustment=True):
    mean_rss = np.asarray(mean_rss, dtype=float)
    return AOS_LOG_FACTOR * np.log1p(mean_rss) if log_adjustment else mean_rss


def aggregate_corners(res, log_adjustment=True, min_sensors=1):
    """One row per visit: recomputed aos_fwhm variants, per-corner z4 deviation, sensor counts."""
    g = res.groupby("visit_id")
    out = pd.DataFrame({"n_sensors": g["rss_period"].count()})
    for name in ("period", "reference", "zero", "noz4"):
        out[f"aos_{name}"] = aos_from_rss(g[f"rss_{name}"].mean(), log_adjustment)
    out["rss_mean_period"] = g["rss_period"].mean()
    out["rss_mean_reference"] = g["rss_reference"].mean()
    out["rss_z4_period"] = g["rss_z4_period"].mean()
    out["rss_z4_reference"] = g["rss_z4_reference"].mean()
    out["rss_noz4"] = g["rss_noz4"].mean()
    per_corner = res.groupby(["visit_id", "corner"])["zdev_4"].mean().unstack("corner")
    for c in CORNERS:
        out[f"z4dev_{c}"] = per_corner[c] if c in per_corner else np.nan
    out["z4dev_mean"] = per_corner.mean(axis=1)
    out.loc[
        out["n_sensors"] < min_sensors, [c for c in out.columns if c != "n_sensors"]
    ] = np.nan
    return out.reset_index()


@dataclass
class QualityCuts:
    fwhm_range: tuple = (0.4, 4.0)
    donut_range: tuple = (0.2, 4.0)
    aos_range: tuple = (0.0, 3.0)
    airmass_max: float = 2.5
    exclude_bands: tuple = ("y",)
    exclude_reasons: tuple = ("filter_change_close_loop",)
    require_corners: bool = False  # drop visits without recomputed AOS values


def apply_quality_cuts(df, cuts=QualityCuts(), verbose=True):
    steps = [
        ("has PSF FWHM", df["psf_fwhm"].notna()),
        (f"PSF FWHM in {cuts.fwhm_range}", df["psf_fwhm"].between(*cuts.fwhm_range)),
        (f"airmass <= {cuts.airmass_max}", df["airmass"] <= cuts.airmass_max),
        (
            f"donut blur present and in {cuts.donut_range}",
            df["donut_blur_fwhm"].between(*cuts.donut_range),
        ),
        (
            f"aos_fwhm present and in {cuts.aos_range}",
            df["aos_fwhm"].between(*cuts.aos_range),
        ),
    ]
    if cuts.exclude_bands:
        steps.append(
            (f"band not in {cuts.exclude_bands}", ~df["band"].isin(cuts.exclude_bands))
        )
    if cuts.exclude_reasons and "observation_reason" in df:
        steps.append(
            (
                f"observation_reason not in {cuts.exclude_reasons}",
                ~df["observation_reason"].isin(cuts.exclude_reasons),
            )
        )
    if cuts.require_corners:
        steps.append(("has recomputed AOS values", df["aos_period"].notna()))
    keep = pd.Series(True, index=df.index)
    rows = []
    for name, mask in steps:
        mask = mask.fillna(False)
        rows.append((name, int((keep & ~mask).sum()), int((keep & mask).sum())))
        keep &= mask
    if verbose:
        print(f"Start: {len(df)} visits")
        print(
            pd.DataFrame(rows, columns=["cut", "removed", "remaining"]).to_string(
                index=False
            )
        )
    return df[keep].copy()

## Statistics and plotting helpers

In [ ]:
def quad_stats(values, min_count=1):
    """sqrt(median(x)) of a *squared* quantity x (arcsec²): n, value (signed sqrt), robust error, 16/84 % as signed sqrt."""
    s = np.asarray(values, dtype=float)
    s = s[np.isfinite(s)]
    row = dict(
        n=len(s), value=np.nan, value_err=np.nan, p16=np.nan, p84=np.nan, median2=np.nan
    )
    if len(s) < max(min_count, 1):
        return row
    med = np.median(s)
    err = 1.2533 * 1.4826 * np.median(np.abs(s - med)) / np.sqrt(len(s))
    row.update(
        value=float(signed_sqrt(med)),
        value_err=err / (2 * np.sqrt(abs(med))) if med != 0 else np.nan,
        p16=float(signed_sqrt(np.percentile(s, 16))),
        p84=float(signed_sqrt(np.percentile(s, 84))),
        median2=med,
    )
    return row


def period_table(df, items, group="period_label", min_count=1):
    """One row per period (plus 'all'): sqrt(median(x²)) of every (column, label) in items, with errors."""
    rows = []
    groups = [("all", df)] + [(k, g) for k, g in df.groupby(group, sort=False)]
    for key, sub in groups:
        row = {
            "period": key,
            "n_visits": len(sub),
            "n_nights": int(sub["day_obs"].nunique()),
        }
        for col, label in items:
            st = quad_stats(
                sub[col] ** 2 if not col.endswith("2") else sub[col],
                min_count=min_count,
            )
            row[label] = st["value"]
            row[f"{label} err"] = st["value_err"]
        rows.append(row)
    return pd.DataFrame(rows).set_index("period")


def nightly_table(df, cols, min_count=5):
    """Per night: count and the 10 / 50 / 90 percentiles of each column (median of squares -> signed sqrt for *2 cols)."""
    rows = []
    for day_obs, sub in df.groupby("day_obs"):
        row = dict(
            day_obs=int(day_obs),
            date=pd.Timestamp(str(int(day_obs))),
            n_visits=len(sub),
            period=int(sub["period"].iloc[0]),
        )
        for c in cols:
            x = sub[c].to_numpy(dtype=float)
            x = x[np.isfinite(x)]
            if len(x) < min_count:
                row[c] = row[f"{c}_p10"] = row[f"{c}_p90"] = np.nan
                continue
            if c.endswith("2"):
                row[c], row[f"{c}_p10"], row[f"{c}_p90"] = (
                    float(signed_sqrt(np.median(x))),
                    float(signed_sqrt(np.percentile(x, 10))),
                    float(signed_sqrt(np.percentile(x, 90))),
                )
            else:
                row[c], row[f"{c}_p10"], row[f"{c}_p90"] = (
                    np.median(x),
                    np.percentile(x, 10),
                    np.percentile(x, 90),
                )
        rows.append(row)
    return pd.DataFrame(rows)


def _break_gaps(nightly, max_gap_days=3):
    """Insert NaN rows after gaps longer than max_gap_days so lines break instead of bridging."""
    nightly = nightly.sort_values("date").reset_index(drop=True)
    gaps = nightly["date"].diff().dt.days > max_gap_days
    if not gaps.any():
        return nightly
    pieces = []
    for i, (_, row) in enumerate(nightly.iterrows()):
        if gaps.iloc[i]:
            blank = row.copy()
            blank[:] = np.nan
            blank["date"] = row["date"] - pd.Timedelta(days=1)
            pieces.append(blank)
        pieces.append(row)
    return pd.DataFrame(pieces).reset_index(drop=True)


def format_time_axis(ax):
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=1))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
    for lab in ax.get_xticklabels():
        lab.set_rotation(30)
        lab.set_ha("right")


def shade_periods(ax, targets, xlim=None, label=True):
    """Vertical lines (and labels along the top) at every change of the z4 target."""
    for k, (day_obs, vals) in enumerate(targets):
        if k == 0:
            continue
        x = pd.Timestamp(str(day_obs))
        ax.axvline(x, color=PERIOD_COLORS[k % len(PERIOD_COLORS)], lw=1.2, ls="--")
        if label:
            ax.annotate(
                target_label(day_obs, vals, index=k),
                (x, 1.0),
                xycoords=("data", "axes fraction"),
                xytext=(3, -3),
                textcoords="offset points",
                fontsize=8,
                va="top",
                ha="left",
                color=PERIOD_COLORS[k % len(PERIOD_COLORS)],
            )


def _pct_series(
    ax, x, med, p10, p90, color, marker, label, offset_days=0.0, ms=4, ls="-"
):
    x = pd.to_datetime(x) + pd.Timedelta(days=offset_days)
    ax.errorbar(
        x,
        med,
        yerr=np.array([med - p10, p90 - med]),
        fmt="none",
        ecolor=color,
        elinewidth=0.9,
        alpha=0.5,
        capsize=0,
    )
    ax.plot(x, med, marker=marker, ms=ms, lw=1.0, ls=ls, color=color, label=label)


def plot_nightly_panels(nightly, panels, targets, height=3.0, show_band=True):
    """Stacked nightly panels. panels = [(ylabel, [(col, label, color, marker, ls), ...], zero_line), ...]."""
    n = len(panels)
    fig, axes = plt.subplots(
        n, 1, figsize=(13, height * n + 0.8), sharex=True, squeeze=False
    )
    axes = axes[:, 0]
    for ax, (ylabel, series, zero_line) in zip(axes, panels):
        offsets = np.linspace(-0.2, 0.2, len(series)) if len(series) > 1 else [0.0]
        for off, (col, label, color, marker, ls) in zip(offsets, series):
            if show_band and f"{col}_p10" in nightly:
                _pct_series(
                    ax,
                    nightly["date"],
                    nightly[col],
                    nightly[f"{col}_p10"],
                    nightly[f"{col}_p90"],
                    color,
                    marker,
                    label,
                    off,
                    ls=ls,
                )
            else:
                ax.plot(
                    nightly["date"],
                    nightly[col],
                    marker=marker,
                    ms=4,
                    lw=1.0,
                    ls=ls,
                    color=color,
                    label=label,
                )
        if zero_line:
            ax.axhline(0, color=INK_MUTED, lw=1, ls="--")
        ax.set_ylabel(ylabel)
        ax.legend(ncol=3, fontsize=9)
        shade_periods(ax, targets, label=ax is axes[0])
    format_time_axis(axes[-1])
    axes[-1].set_xlabel("night (day_obs)")
    fig.align_ylabels(axes)
    return fig


def plot_visit_scatter(
    df, series, targets, ylabel, ylim=None, height=3.2, nightly_line=True, alpha=0.25
):
    """Visit-by-visit scatter against time, one panel per (col, label, color), coloured by period, nightly median over it."""
    n = len(series)
    fig, axes = plt.subplots(
        n, 1, figsize=(13, height * n + 0.8), sharex=True, squeeze=False
    )
    axes = axes[:, 0]
    for ax, (col, label, color) in zip(axes, series):
        for p, sub in df.groupby("period"):
            ax.scatter(
                sub["time"],
                sub[col],
                s=5,
                alpha=alpha,
                lw=0,
                color=PERIOD_COLORS[p % len(PERIOD_COLORS)],
                rasterized=True,
            )
        if nightly_line:
            nightly = _break_gaps(nightly_table(df, [col]))
            ax.plot(
                nightly["date"] + pd.Timedelta(hours=12),
                nightly[col],
                color=SURFACE,
                lw=3.5,
            )
            ax.plot(
                nightly["date"] + pd.Timedelta(hours=12),
                nightly[col],
                color=INK,
                lw=1.6,
                marker="o",
                ms=3,
                label="nightly median",
            )
        ax.set_ylabel(ylabel)
        ax.set_title(label, fontsize=11, loc="left")
        if ylim:
            ax.set_ylim(*ylim)
        shade_periods(ax, targets, label=ax is axes[0])
    handles = [
        plt.Line2D(
            [],
            [],
            marker="o",
            ls="",
            color=PERIOD_COLORS[k % len(PERIOD_COLORS)],
            label=target_label(d, v, index=k),
        )
        for k, (d, v) in enumerate(targets)
    ]
    if nightly_line:
        handles.append(
            plt.Line2D(
                [], [], color=INK, lw=1.6, marker="o", ms=3, label="nightly median"
            )
        )
    fig.legend(
        handles=handles,
        fontsize=9,
        ncol=len(handles),
        loc="upper center",
        bbox_to_anchor=(0.5, 1.0),
    )
    format_time_axis(axes[-1])
    axes[-1].set_xlabel("time")
    fig.align_ylabels(axes)
    fig.tight_layout(rect=(0, 0, 1, 0.97))
    return fig


def plot_corner_z4(res_visits, targets, reference_index, ylim=(-0.8, 0.6)):
    """Per corner: visit-by-visit z4 deviation (um) with the target in force (steps) and the reference target (dashed)."""
    fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True, sharey=True)
    tmin, tmax = res_visits["time"].min(), res_visits["time"].max()
    for ax, corner in zip(axes.flat, CORNERS):
        col = f"z4dev_{corner}"
        for p, sub in res_visits.groupby("period"):
            ax.scatter(
                sub["time"],
                sub[col],
                s=4,
                alpha=0.25,
                lw=0,
                color=PERIOD_COLORS[p % len(PERIOD_COLORS)],
                rasterized=True,
            )
        nightly = _break_gaps(nightly_table(res_visits, [col]))
        ax.plot(
            nightly["date"] + pd.Timedelta(hours=12),
            nightly[col],
            color=SURFACE,
            lw=3.5,
        )
        ax.plot(
            nightly["date"] + pd.Timedelta(hours=12),
            nightly[col],
            color=INK,
            lw=1.5,
            marker="o",
            ms=3,
            label="nightly median",
        )
        # target in force, as steps
        xs, ys = [], []
        for k, (day_obs, vals) in enumerate(targets):
            x0 = max(pd.Timestamp(str(day_obs)), tmin) if k else tmin
            x1 = pd.Timestamp(str(targets[k + 1][0])) if k + 1 < len(targets) else tmax
            xs += [x0, x1]
            ys += [vals[corner], vals[corner]]
        ax.plot(xs, ys, color=SERIES_COLORS[7], lw=2.0, label="z4 target in force")
        ax.axhline(
            targets[reference_index][1][corner],
            color=SERIES_COLORS[7],
            lw=1.2,
            ls="--",
            label="reference target",
        )
        ax.axhline(0, color=INK_MUTED, lw=0.8, ls=":")
        ax.set_title(f"{corner}  (SW0 + SW1 mean)", fontsize=11, loc="left")
        ax.set_ylim(*ylim)
        shade_periods(ax, targets, label=False)
    axes[0, 0].legend(fontsize=8, loc="lower left")
    for ax in axes[:, 0]:
        ax.set_ylabel("z4 - intrinsic (um)")
    for ax in axes[-1]:
        format_time_axis(ax)
        ax.set_xlabel("time")
    return fig


def plot_period_hexbin(
    df,
    xcol,
    ycol,
    targets,
    xlabel,
    ylabel,
    lim=(0.0, 1.5),
    gridsize=40,
    one_to_one=True,
    ylim=None,
):
    """One hexbin panel per period of y vs x with the binned median."""
    periods = sorted(df["period"].dropna().unique())
    n = len(periods)
    fig, axes = plt.subplots(
        1, n, figsize=(4.8 * n, 4.6), squeeze=False, layout="constrained"
    )
    xbins = np.linspace(lim[0], lim[1], 21)
    ylim = ylim or lim
    hb = None
    for ax, p in zip(axes.flat, periods):
        sub = df[df["period"] == p]
        ok = sub[xcol].notna() & sub[ycol].notna()
        if ok.sum():
            hb = ax.hexbin(
                sub.loc[ok, xcol],
                sub.loc[ok, ycol],
                gridsize=gridsize,
                extent=(*lim, *ylim),
                cmap=SEQUENTIAL_BLUE,
                norm=LogNorm(vmin=1),
                mincnt=1,
                linewidths=0.2,
                edgecolors="face",
            )
        st = binned_median(sub.loc[ok, xcol], sub.loc[ok, ycol], xbins)
        if one_to_one:
            ax.plot(lim, lim, color=INK_MUTED, lw=1, ls="--", label="1:1")
        ax.plot(st["center"], st["median"], color=SURFACE, lw=4)
        ax.plot(
            st["center"],
            st["median"],
            color=SERIES_COLORS[1],
            lw=2,
            marker="o",
            ms=4,
            label="binned median",
        )
        ax.set_title(
            f"{target_label(*targets[int(p)], index=int(p))}   (n = {int(ok.sum())})",
            fontsize=9.5,
        )
        ax.set_xlabel(xlabel)
        ax.set_ylabel(ylabel)
        ax.set_xlim(*lim)
        ax.set_ylim(*ylim)
    axes[0, 0].legend(loc="upper left", fontsize=9)
    if hb is not None:
        cbar = fig.colorbar(hb, ax=list(axes.flat), pad=0.02, shrink=0.9)
        cbar.set_label("visits per hexagon")
    return fig


def binned_median(x, y, bins, min_count=10):
    """Median, 16/84 percentiles and robust median error of y in bins of x (y may be a squared quantity)."""
    x, y = np.asarray(x, dtype=float), np.asarray(y, dtype=float)
    ok = np.isfinite(x) & np.isfinite(y)
    x, y = x[ok], y[ok]
    edges = np.asarray(bins, dtype=float)
    idx = np.digitize(x, edges) - 1
    rows = []
    for i in range(len(edges) - 1):
        sel = y[idx == i]
        row = dict(
            lo=edges[i],
            hi=edges[i + 1],
            center=0.5 * (edges[i] + edges[i + 1]),
            n=len(sel),
            median=np.nan,
            p16=np.nan,
            p84=np.nan,
            median_err=np.nan,
        )
        if len(sel) >= min_count:
            med = np.median(sel)
            row.update(
                median=med,
                p16=np.percentile(sel, 16),
                p84=np.percentile(sel, 84),
                median_err=1.2533
                * 1.4826
                * np.median(np.abs(sel - med))
                / np.sqrt(len(sel)),
            )
        rows.append(row)
    return pd.DataFrame(rows)


def plot_binned_by_period(
    df,
    xcol,
    ycol,
    bins,
    targets,
    xlabel,
    ylabel,
    sqrt_output=False,
    zero_line=True,
    ax=None,
):
    """Binned median of y vs x, one line per period (signed sqrt of the median when sqrt_output)."""
    fig, ax = (ax.figure, ax) if ax is not None else plt.subplots(figsize=(9, 5))
    for p, sub in df.groupby("period"):
        st = binned_median(sub[xcol], sub[ycol], bins)
        y = signed_sqrt(st["median"]) if sqrt_output else st["median"]
        yerr = (
            st["median_err"] / (2 * np.sqrt(np.abs(st["median"])))
            if sqrt_output
            else st["median_err"]
        )
        color = PERIOD_COLORS[int(p) % len(PERIOD_COLORS)]
        ax.errorbar(
            st["center"],
            y,
            yerr=yerr,
            color=color,
            marker=SERIES_MARKERS[int(p) % 8],
            ms=5,
            capsize=2,
            lw=1.8,
            label=f"{target_label(*targets[int(p)], index=int(p))} (n = {len(sub)})",
        )
    if zero_line:
        ax.axhline(0, color=INK_MUTED, lw=1, ls="--")
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_xlim(float(bins[0]), float(bins[-1]))
    ax.legend(fontsize=8)
    return fig


def plot_histograms_by_period(df, col, bins, targets, xlabel, ax=None, density=True):
    """Step histogram per period with the median marked (25 / 75 % quoted in the legend)."""
    fig, ax = (ax.figure, ax) if ax is not None else plt.subplots(figsize=(9, 5))
    for p, sub in df.groupby("period"):
        d = sub[col].dropna()
        if not len(d):
            continue
        color = PERIOD_COLORS[int(p) % len(PERIOD_COLORS)]
        p25, p50, p75 = d.quantile([0.25, 0.5, 0.75])
        ax.hist(
            d.clip(bins[0], bins[-1]),
            bins=bins,
            histtype="step",
            lw=2,
            color=color,
            density=density,
            label=f"{target_label(*targets[int(p)], index=int(p))}: median {p50:.3f} [{p25:.3f}, {p75:.3f}] (n = {len(d)})",
        )
        ax.axvline(p50, color=color, lw=1.2, ls=":")
    ax.set_xlabel(xlabel)
    ax.set_ylabel("fraction of visits" if density else "visits")
    ax.set_xlim(float(bins[0]), float(bins[-1]))
    ax.legend(fontsize=8)
    return fig

## Parameters and data

Two ConsDB pulls: one row per visit (`visit1` + `visit1_quicklook` + `exposure`, as in the companion notebooks,
plus the physical rotator angle) and one row per (visit, wavefront sensor) with `z4 … z28`.

In [ ]:
start_dayobs = parse_day_obs(start_dayobs)
end_dayobs = parse_day_obs(end_dayobs)
if start_dayobs > end_dayobs:
    raise ValueError(
        f"start_dayobs ({start_dayobs}) must be <= end_dayobs ({end_dayobs})"
    )
programs = parse_programs(science_programs)
TARGETS = parse_z4_targets(z4_targets)
reference_target_dayobs = parse_day_obs(reference_target_dayobs)
REFERENCE_INDEX = [d for d, _ in TARGETS].index(reference_target_dayobs)
subtract_intrinsics = parse_bool(subtract_intrinsics)
aos_log_adjustment = parse_bool(aos_log_adjustment)
max_noll_index = int(max_noll_index)
EXCLUDE_BANDS = parse_bands(exclude_bands)
use_cache = parse_bool(use_cache)
intrinsic_rotation_sign = str(intrinsic_rotation_sign).strip().lower()
if intrinsic_rotation_sign not in ("auto", "+1", "-1", "1"):
    raise ValueError("intrinsic_rotation_sign must be 'auto', '+1' or '-1'")

cache_tag = f"{CACHE_VERSION}_{start_dayobs}_{end_dayobs}_{'_'.join(p.replace('BLOCK-', '') for p in programs)}"
cache_dir = (
    os.path.join(NOTEBOOK_DIR, "cache") if (use_cache and not_times_square) else None
)

print(
    f"Nights {start_dayobs}-{end_dayobs}; programs {programs}; bands excluded {EXCLUDE_BANDS}"
)
for k, (d, v) in enumerate(TARGETS):
    print(
        f"  P{k}: {target_description(d, v)}"
        + ("   <- reference target" if k == REFERENCE_INDEX else "")
    )

In [ ]:
consdb = get_consdb_client()
raw = cached(
    cache_dir and os.path.join(cache_dir, f"visits_{cache_tag}.parquet"),
    lambda: fetch_visits(consdb, start_dayobs, end_dayobs, programs),
    use_cache,
)
corners = cached(
    cache_dir and os.path.join(cache_dir, f"corner_zernikes_{cache_tag}.parquet"),
    lambda: fetch_corner_zernikes(consdb, start_dayobs, end_dayobs, programs),
    use_cache,
)
if "sensor" not in corners and len(corners):
    corners["sensor"] = corners["detector"].astype(int).map(CORNER_DETECTORS)
    corners["corner"] = corners["sensor"].str[:3]

print(
    f"{len(raw)} visits on {raw['day_obs'].nunique()} nights; {len(corners)} wavefront-sensor rows "
    f"on {corners['visit_id'].nunique()} visits"
)
if len(corners):
    print("sensor rows with a z4 value, per sensor:")
    print(corners.groupby("sensor")["z4"].count().to_string())
else:
    print(
        "WARNING: no corner Zernikes came back; the recomputed AOS section will be empty. Detectors with any "
        "Zernike values in the range:"
    )
    print(
        consdb.query(
            f"SELECT c.detector, COUNT(q.z4) AS n_z4 FROM cdb_lsstcam.ccdvisit1 AS c "
            f"JOIN cdb_lsstcam.ccdvisit1_quicklook AS q ON q.ccdvisit_id = c.ccdvisit_id "
            f"WHERE c.day_obs >= {start_dayobs} AND c.day_obs <= {end_dayobs} GROUP BY c.detector ORDER BY n_z4 DESC"
        ).head(20)
    )

In [ ]:
# What came back and how complete the key columns are.
if len(raw):
    print(pd.crosstab(raw["science_program"], raw["band"], margins=True))
print("\nfraction of visits with a value:")
print(
    raw[["psf_sigma_median", "donut_blur_fwhm", "aos_fwhm", "physical_rotator_angle"]]
    .notna()
    .mean()
    .round(3)
)

## Derived quantities, periods and quality cuts

Per visit: the delivered PSF FWHM, $\theta_\mathrm{alt}^2 = \mathrm{FWHM}^2 - \mathrm{donut\_blur}^2$,
$\mathrm{aos\_fwhm}^2$ and their difference $\mathrm{aos\_fwhm}^2 - \theta_\mathrm{alt}^2$ (positive when the
AOS term is the larger one), and the period, i.e. the $z_4$ target set in force on that night.

In [ ]:
visits = raw.copy()
visits["psf_fwhm"] = (
    visits["psf_sigma_median"].astype(float) * SIGMA_TO_FWHM * PLATESCALE
)
visits["time"] = pd.to_datetime(
    visits["exp_midpt"], utc=True, errors="coerce"
).dt.tz_localize(None)
visits["psf2"] = visits["psf_fwhm"] ** 2
visits["donut2"] = visits["donut_blur_fwhm"].astype(float) ** 2
visits["aos2"] = visits["aos_fwhm"].astype(float) ** 2
visits["alt2"] = visits["psf2"] - visits["donut2"]  # optics from PSF and donut blur
visits["alt"] = signed_sqrt(visits["alt2"])
visits["aos_minus_alt2"] = (
    visits["aos2"] - visits["alt2"]
)  # > 0: aos_fwhm larger than sqrt(FWHM² - donut²)
visits["period"] = assign_periods(visits["day_obs"].to_numpy(), TARGETS)
visits["period_label"] = [
    target_label(*TARGETS[p], index=int(p)) for p in visits["period"]
]

cuts = QualityCuts(
    fwhm_range=(0.4, 4.0),
    donut_range=(0.2, 4.0),
    aos_range=(0.0, 3.0),
    airmass_max=2.5,
    exclude_bands=EXCLUDE_BANDS,
    exclude_reasons=("filter_change_close_loop",),
)
sel = apply_quality_cuts(visits, cuts)
print(f"\n{len(sel)} visits kept on {sel['day_obs'].nunique()} nights; per period:")
print(
    sel.groupby("period_label", sort=False)
    .agg(
        n_visits=("visit_id", "size"),
        n_nights=("day_obs", "nunique"),
        first=("day_obs", "min"),
        last=("day_obs", "max"),
    )
    .to_string()
)

## 1. Stored `aos_fwhm` against $\sqrt{\mathrm{FWHM}^2 - \mathrm{donut\_blur}^2}$

Headline per period: the delivered FWHM, the donut blur, the two optics estimates and their difference, all as
$\sqrt{\mathrm{median}(\cdot^2)}$. The last column is the fraction of visits in which `aos_fwhm` exceeds
$\theta_\mathrm{alt}$.

In [ ]:
HEADLINE_ITEMS = [
    ("psf_fwhm", "FWHM"),
    ("donut_blur_fwhm", "donut blur"),
    ("alt2", "sqrt(FWHM² - donut²)"),
    ("aos_fwhm", "aos_fwhm"),
    ("aos_minus_alt2", "sqrt(aos² - (FWHM² - donut²))"),
]
headline = period_table(sel, HEADLINE_ITEMS)
headline["frac aos > alt"] = pd.Series(
    {
        "all": (sel["aos_minus_alt2"] > 0).mean(),
        **{
            k: (g["aos_minus_alt2"] > 0).mean()
            for k, g in sel.groupby("period_label", sort=False)
        },
    }
)
headline.round(3)

In [ ]:
NIGHTLY_COLS = [
    "psf_fwhm",
    "donut_blur_fwhm",
    "aos_fwhm",
    "alt2",
    "aos_minus_alt2",
    "psf2",
    "donut2",
]
nightly = _break_gaps(nightly_table(sel, NIGHTLY_COLS))

fig = plot_nightly_panels(
    nightly,
    [
        (
            "per-visit value (arcsec)",
            [
                ("psf_fwhm", "delivered PSF FWHM", INK, "o", "-"),
                ("donut_blur_fwhm", "donut blur", SERIES_COLORS[2], "s", "-"),
            ],
            False,
        ),
        (
            "optics term (arcsec)",
            [
                (
                    "alt2",
                    "sqrt(FWHM² - donut²)  [nightly sqrt(median)]",
                    SERIES_COLORS[1],
                    "^",
                    "-",
                ),
                ("aos_fwhm", "aos_fwhm (ConsDB)", INK, "o", "-"),
            ],
            False,
        ),
        (
            "signed sqrt(median(aos² - alt²)) (arcsec)",
            [
                (
                    "aos_minus_alt2",
                    "aos_fwhm² - (FWHM² - donut²): > 0 when aos_fwhm is the larger",
                    SERIES_COLORS[6],
                    "D",
                    "-",
                )
            ],
            True,
        ),
    ],
    TARGETS,
)
fig.suptitle(
    "marker: nightly median; bar: 10th-90th percentile of the per-visit values",
    fontsize=10,
    y=1.0,
)
plt.show()

### 1a. What moved: FWHM or donut blur?

The change of $\mathrm{median}(\mathrm{FWHM}^2 - \mathrm{donut\_blur}^2)$ from one period to the next is split
into the change of $\mathrm{median}(\mathrm{FWHM}^2)$ and the change of $\mathrm{median}(\mathrm{donut\_blur}^2)$
(the medians of the two inputs are taken separately, so the split is approximate but the sign and the size
of each part are what matter). The same is done for $\mathrm{aos\_fwhm}^2$ so the two optics estimates can be
compared directly. The second figure removes the atmosphere: $\mathrm{FWHM}^2 - \mathrm{donut\_blur}^2$ is
binned by donut blur, period by period. If the curves sit on top of each other the periods differ only through
the seeing; if they are offset, the optics (or the camera / dome) term itself changed.

In [ ]:
def period_change_table(df, cols, group="period_label"):
    meds = df.groupby(group, sort=False)[cols].median()
    meds.loc["all"] = df[cols].median()
    diffs = meds.drop(index="all").diff().iloc[1:]
    diffs.index = [f"{a} -> {b}" for a, b in zip(meds.index[:-2], meds.index[1:-1])]
    return meds, diffs


SQ_COLS = ["psf2", "donut2", "alt2", "aos2", "aos_minus_alt2"]
medians, changes = period_change_table(sel, SQ_COLS)
print("medians of the squared quantities (arcsec²):")
print(medians.round(4).to_string())
print("\nchange from one period to the next (arcsec²):  alt2 = psf2 - donut2")
changes = changes.rename(
    columns={
        "psf2": "d median(FWHM²)",
        "donut2": "d median(donut²)",
        "alt2": "d median(FWHM² - donut²)",
        "aos2": "d median(aos²)",
        "aos_minus_alt2": "d median(aos² - alt²)",
    }
)
print(changes.round(4).to_string())

fig, ax = plt.subplots(figsize=(10, 4.8))
labels = [
    "d median(FWHM²)",
    "d median(donut²)",
    "d median(FWHM² - donut²)",
    "d median(aos²)",
]
colors = [INK, SERIES_COLORS[2], SERIES_COLORS[1], SERIES_COLORS[0]]
x = np.arange(len(changes))
w = 0.8 / len(labels)
for k, (lab, col) in enumerate(zip(labels, colors)):
    ax.bar(
        x + (k - (len(labels) - 1) / 2) * w, changes[lab], width=w, color=col, label=lab
    )
ax.axhline(0, color=INK_MUTED, lw=1)
ax.set_xticks(x)
ax.set_xticklabels([s.replace(" -> ", "\n-> ") for s in changes.index], fontsize=8)
ax.set_ylabel("change of the median (arcsec²)")
ax.set_title(
    "period-to-period change: does FWHM² - donut² move because of FWHM or because of the donut blur?"
)
ax.legend(fontsize=9)
plt.show()

In [ ]:
donut_bins = np.arange(0.4, 2.01, 0.1)
fig, axes = plt.subplots(1, 3, figsize=(18, 5.2))
plot_binned_by_period(
    sel,
    "donut_blur_fwhm",
    "alt2",
    donut_bins,
    TARGETS,
    "donut blur (arcsec)",
    "sqrt(median FWHM² - donut²) (arcsec)",
    sqrt_output=True,
    ax=axes[0],
)
axes[0].set_title("optics from PSF and donut, at fixed donut blur")
plot_binned_by_period(
    sel,
    "donut_blur_fwhm",
    "aos2",
    donut_bins,
    TARGETS,
    "donut blur (arcsec)",
    "sqrt(median aos_fwhm²) (arcsec)",
    sqrt_output=True,
    ax=axes[1],
)
axes[1].set_title("aos_fwhm, at fixed donut blur")
plot_histograms_by_period(
    sel,
    "donut_blur_fwhm",
    np.arange(0.3, 2.5, 0.04),
    TARGETS,
    "donut blur (arcsec)",
    ax=axes[2],
)
axes[2].set_title("donut blur distribution per period")
fig.tight_layout()
plt.show()

fig = plot_period_hexbin(
    sel,
    "donut_blur_fwhm",
    "psf_fwhm",
    TARGETS,
    "donut blur (arcsec)",
    "delivered PSF FWHM (arcsec)",
    lim=(0.4, 2.2),
)
plt.show()

### 1b. Visit by visit

The stored `aos_fwhm`, $\theta_\mathrm{alt}$ and their quadrature difference for every visit, coloured by period.

In [ ]:
fig = plot_visit_scatter(
    sel,
    [
        ("aos_fwhm", "aos_fwhm (ConsDB)", INK),
        ("alt", "signed sqrt(FWHM² - donut²)", SERIES_COLORS[1]),
        ("aos_minus_alt2", "aos² - (FWHM² - donut²)  (arcsec²)", SERIES_COLORS[6]),
    ],
    TARGETS,
    ylabel="arcsec",
)
fig.axes[2].set_ylabel("arcsec²")
fig.axes[2].axhline(0, color=INK_MUTED, lw=1, ls="--")
fig.axes[0].set_ylim(0, 1.2)
fig.axes[1].set_ylim(-0.6, 1.2)
fig.axes[2].set_ylim(-0.8, 0.8)
plt.show()

fig = plot_period_hexbin(
    sel,
    "alt",
    "aos_fwhm",
    TARGETS,
    "signed sqrt(FWHM² - donut²) (arcsec)",
    "aos_fwhm (ConsDB) (arcsec)",
    lim=(-0.2, 1.2),
)
plt.show()

## 2. `aos_fwhm` recomputed from the corner Zernikes

The per-sensor Zernikes are turned into the residual PSF width for each target variant. First the convention
check: the recomputation relative to the target in force at the time is compared with the stored column for
each candidate convention (rotation sign of the intrinsics, or no intrinsic subtraction), and the one with the
smallest median absolute difference is kept unless `intrinsic_rotation_sign` fixes it. The table reports the
agreement per period as well, so a period in which the stored column was computed against a *different* target
stands out.

In [ ]:
BANDS_PRESENT = sorted(
    set(str(b).strip()[0] for b in sel["band"].dropna().unique())
    & set(OFC_WAVELENGTH_UM)
)
factors = psf_width_per_zernike(Z_MIN, max_noll_index)
if not np.allclose(factors, PSF_WIDTH_PER_ZERNIKE[: len(factors)], rtol=1e-4):
    print(
        "WARNING: galsim conversion factors differ from the stored table:",
        factors.round(4),
    )
print("PSF width per um of z4..z8 (arcsec/um):", factors[:5].round(4))

tables = (
    load_intrinsic_tables(BANDS_PRESENT, cache_dir) if subtract_intrinsics else None
)

candidates = {}
if subtract_intrinsics:
    if intrinsic_rotation_sign == "auto":
        candidates = {"+rot": 1.0, "-rot": -1.0}
    else:
        candidates = {
            f"{float(intrinsic_rotation_sign):+g}rot": float(intrinsic_rotation_sign)
        }
else:
    candidates = {"no intrinsics": None}

corner_sel = corners[corners["visit_id"].isin(sel["visit_id"])]
results, checks = {}, []
for name, sign in candidates.items():
    res = corner_residuals(
        corner_sel,
        sel,
        TARGETS,
        REFERENCE_INDEX,
        tables,
        rotation_sign=sign or 1.0,
        subtract_intrinsics=sign is not None,
        jmax=max_noll_index,
    )
    agg = aggregate_corners(res, aos_log_adjustment)
    m = sel[["visit_id", "aos_fwhm", "period_label"]].merge(
        agg[["visit_id", "aos_period"]], on="visit_id"
    )
    d = m["aos_period"] - m["aos_fwhm"]
    row = dict(
        convention=name,
        n=int(d.notna().sum()),
        median_abs_diff=float(d.abs().median()),
        median_diff=float(d.median()),
        p16=float(d.quantile(0.16)),
        p84=float(d.quantile(0.84)),
    )
    for lab, g in m.groupby("period_label", sort=False):
        row[f"median |diff| {lab}"] = float(
            (g["aos_period"] - g["aos_fwhm"]).abs().median()
        )
    checks.append(row)
    results[name] = (res, agg)
checks = pd.DataFrame(checks).set_index("convention")
print("recomputed (period target) minus stored aos_fwhm, arcsec:")
print(checks.round(4).T.to_string())
best = checks["median_abs_diff"].idxmin()
print(f"\nusing convention: {best}")
res, agg = results[best]

In [ ]:
sel = sel.drop(
    columns=[
        c
        for c in sel.columns
        if c.startswith(
            (
                "aos_period",
                "aos_reference",
                "aos_zero",
                "aos_noz4",
                "rss_",
                "z4dev_",
                "n_sensors",
            )
        )
    ]
)
sel = sel.merge(agg, on="visit_id", how="left")
for name in ("period", "reference", "zero", "noz4"):
    sel[f"aos_{name}2"] = sel[f"aos_{name}"] ** 2
    sel[f"aos_{name}_minus_alt2"] = sel[f"aos_{name}2"] - sel["alt2"]
sel["recalc_minus_stored"] = sel["aos_period"] - sel["aos_fwhm"]
print(
    f"visits with recomputed AOS values: {sel['aos_period'].notna().sum()} of {len(sel)} "
    f"({sel['aos_period'].notna().mean():.1%}); sensors per visit:"
)
print(sel["n_sensors"].value_counts(dropna=False).sort_index().to_string())
print(
    "\nmissing rotator angle (no intrinsics possible):",
    int(sel["physical_rotator_angle"].isna().sum()),
)

### 2a. Validation: period-target recomputation vs the stored column

Visit by visit, per period. A tight 1:1 relation means the stored `aos_fwhm` was computed with the target of
that period (and the same intrinsics); an offset in one period means the stored values there used another target.

In [ ]:
fig = plot_period_hexbin(
    sel,
    "aos_fwhm",
    "aos_period",
    TARGETS,
    "aos_fwhm (ConsDB) (arcsec)",
    "recomputed, period target (arcsec)",
    lim=(0.0, 1.2),
)
plt.show()

fig = plot_histograms_by_period(
    sel,
    "recalc_minus_stored",
    np.arange(-0.3, 0.3001, 0.005),
    TARGETS,
    "recomputed (period target) - stored aos_fwhm (arcsec)",
)
plt.show()

### 2b. How the AOS was converging: per-corner $z_4$ deviation

The measured minus intrinsic $z_4$ on each corner for every visit, with the target in force (solid steps) and the
reference target (dashed). The table gives, per period and corner, the median deviation and the median residual
to each target, in µm, and the corresponding PSF-width term $c_4 \times$ residual.

In [ ]:
fig = plot_corner_z4(sel.dropna(subset=["z4dev_mean"]), TARGETS, REFERENCE_INDEX)
plt.show()

rows = []
for (lab, p), g in sel.dropna(subset=["z4dev_mean"]).groupby(
    ["period_label", "period"], sort=False
):
    for c in CORNERS:
        d = g[f"z4dev_{c}"].dropna()
        t_p, t_r = TARGETS[int(p)][1][c], TARGETS[REFERENCE_INDEX][1][c]
        rows.append(
            dict(
                period=lab,
                corner=c,
                n=len(d),
                median_z4dev_um=d.median(),
                p25=d.quantile(0.25),
                p75=d.quantile(0.75),
                target_period=t_p,
                resid_to_period_um=(d - t_p).median(),
                target_reference=t_r,
                resid_to_reference_um=(d - t_r).median(),
                fwhm_term_period=factors[0] * abs((d - t_p).median()),
                fwhm_term_reference=factors[0] * abs((d - t_r).median()),
            )
        )
z4_table = pd.DataFrame(rows).set_index(["period", "corner"])
z4_table.round(4)

### 2c. The recomputed optics term per period, and what drives it

`rss_z4_*` is the $z_4$ term alone ($c_4 |z_4 - z_4^\mathrm{intr} - y_{2,4}|$, averaged over sensors) and
`rss_noz4` the quadrature sum of the higher-order modes; `aos_*` are the full recomputed values (with the
logarithmic adjustment when `aos_log_adjustment` is on). All as $\sqrt{\mathrm{median}(\cdot^2)}$ per period.

In [ ]:
VARIANT_ITEMS = [(k, v[0]) for k, v in VARIANT_STYLE.items()] + [
    ("rss_z4_period", "z4 term, period target"),
    ("rss_z4_reference", "z4 term, reference target"),
    ("rss_noz4", "higher-order modes (no z4)"),
]
variants = period_table(sel.dropna(subset=["aos_period"]), VARIANT_ITEMS)
variants[[c for c in variants.columns if not c.endswith(" err")]].round(3)

In [ ]:
nightly = _break_gaps(
    nightly_table(
        sel,
        [
            "aos_fwhm",
            "aos_period",
            "aos_reference",
            "aos_zero",
            "aos_noz4",
            "rss_z4_period",
            "rss_z4_reference",
            "rss_noz4",
            "alt2",
            "aos_period_minus_alt2",
            "aos_reference_minus_alt2",
            "aos_minus_alt2",
        ],
    )
)
fig = plot_nightly_panels(
    nightly,
    [
        (
            "aos_fwhm variants (arcsec)",
            [(k, *v[:1], v[1], v[2], v[3]) for k, v in VARIANT_STYLE.items()],
            False,
        ),
        (
            "RSS parts (arcsec)",
            [
                ("rss_z4_period", "z4 term, period target", SERIES_COLORS[0], "s", "-"),
                (
                    "rss_z4_reference",
                    "z4 term, reference target",
                    SERIES_COLORS[1],
                    "^",
                    "-",
                ),
                ("rss_noz4", "higher-order modes", SERIES_COLORS[6], "v", "--"),
            ],
            False,
        ),
    ],
    TARGETS,
    show_band=False,
)
plt.show()

### 2d. Does the reference-target `aos_fwhm` explain $\sqrt{\mathrm{FWHM}^2 - \mathrm{donut\_blur}^2}$?

For each variant, the signed $\sqrt{\mathrm{median}(\mathrm{aos}^2 - (\mathrm{FWHM}^2 - \mathrm{donut\_blur}^2))}$
per period. The variant that explains the delivered data is the one whose difference stays flat across the target
changes; if that is the reference-target variant, the crossover seen in the stored column is an artefact of the
moving $z_4$ target rather than a change in the optics.

In [ ]:
DIFF_ITEMS = [
    ("aos_minus_alt2", "stored aos_fwhm"),
    ("aos_period_minus_alt2", "recomputed, period target"),
    ("aos_reference_minus_alt2", "recomputed, reference target"),
    ("aos_zero_minus_alt2", "recomputed, zero target"),
    ("aos_noz4_minus_alt2", "recomputed, without z4"),
]
diff_table = period_table(sel.dropna(subset=["aos_period"]), DIFF_ITEMS)
print(
    "signed sqrt(median(aos² - (FWHM² - donut²))) per period, arcsec  (> 0: the AOS term is the larger one)"
)
diff_table[[c for c in diff_table.columns if not c.endswith(" err")]].round(3)

In [ ]:
fig = plot_nightly_panels(
    nightly,
    [
        (
            "optics term (arcsec)",
            [
                ("alt2", "sqrt(FWHM² - donut²)", SERIES_COLORS[1], "^", "-"),
                ("aos_fwhm", "aos_fwhm (ConsDB)", INK, "o", "-"),
                (
                    "aos_reference",
                    "recomputed, reference target",
                    SERIES_COLORS[0],
                    "s",
                    "-",
                ),
            ],
            False,
        ),
        (
            "signed sqrt(median(aos² - alt²)) (arcsec)",
            [
                ("aos_minus_alt2", "stored aos_fwhm", INK, "o", "-"),
                (
                    "aos_period_minus_alt2",
                    "recomputed, period target",
                    SERIES_COLORS[0],
                    "s",
                    "-",
                ),
                (
                    "aos_reference_minus_alt2",
                    "recomputed, reference target",
                    SERIES_COLORS[1],
                    "^",
                    "-",
                ),
            ],
            True,
        ),
    ],
    TARGETS,
    show_band=False,
)
plt.show()

fig = plot_period_hexbin(
    sel,
    "alt",
    "aos_reference",
    TARGETS,
    "signed sqrt(FWHM² - donut²) (arcsec)",
    "recomputed aos_fwhm, reference target (arcsec)",
    lim=(-0.2, 1.2),
)
plt.show()

### 2e. Visit by visit: the recomputed AOS values across the periods

In [ ]:
fig = plot_visit_scatter(
    sel.dropna(subset=["aos_period"]),
    [
        ("aos_period", "recomputed aos_fwhm, period target", SERIES_COLORS[0]),
        ("aos_reference", "recomputed aos_fwhm, reference target", SERIES_COLORS[1]),
        ("aos_noz4", "recomputed, higher-order modes only (no z4)", SERIES_COLORS[6]),
        (
            "z4dev_mean",
            "z4 - intrinsic, mean of the four corners (um)",
            SERIES_COLORS[3],
        ),
    ],
    TARGETS,
    ylabel="arcsec",
)
for ax in fig.axes[:3]:
    ax.set_ylim(0, 1.2)
fig.axes[3].set_ylabel("um")
fig.axes[3].set_ylim(-0.8, 0.6)
fig.axes[3].axhline(0, color=INK_MUTED, lw=0.8, ls=":")
plt.show()

fig = plot_histograms_by_period(
    sel,
    "aos_reference",
    np.arange(0, 1.2, 0.02),
    TARGETS,
    "recomputed aos_fwhm, reference target (arcsec)",
)
plt.show()

## Notes

* `aos_fwhm` in the ConsDB is filled by Rapid Analysis with the $y_2$ offsets of the `ts_ofc` version deployed
  at the time, so the stored column is only comparable across periods if that version was updated in step with
  the closed-loop target; the validation in 2a shows where it was not. The recomputed variants use the targets
  given in `z4_targets` explicitly.
* The intrinsic Zernikes come from the `ts_ofc` policy tables at the pinned commit above, evaluated as
  `lsst.ts.ofc.utils.get_intrinsic_zernikes` does. The current `ts_wep` takes them from a butler calibration
  (`IntrinsicZernikes`) instead; small differences between the two are absorbed in the validation offset.
* The Zernike-to-PSF-width conversion is linear and starts to over-estimate the degradation when the RSS exceeds
  about 0.2″ (see the `convertZernikesToPsfWidth` docstring); the 1.06 ln(1 + x) adjustment partly compensates.
* The higher-order modes are identical in every variant, so differences between variants are entirely the $z_4$
  term; `rss_noz4` is the floor the optics term cannot go below whatever the defocus target.
* Only $y_{2,4}$ is varied here. If other $y_2$ components were ever non-zero, add them to the parsing.